In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import RobustScaler

parent_dir = os.path.abspath(os.path.join(os.path.dirname('utils.py'), '..'))
sys.path.append(parent_dir)
import utils as ut  # noqa: E402

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IN_FEATURES = ['beta', 'Av', 'F115', 'F150', 'F277', 'F444']
OUT_FEATURES = ['flag']
SEED = 42

# HYPERPARAMETERS
EPOCHS = 10000
NUM_TRIALS = 20
PATIENCE = 30
OPTIMIZE = True

if OPTIMIZE:
    COMMENT = input('Enter OPTIMIZATION comment: ')

In [18]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ut.set_seeds(SEED)

In [19]:
cwd = Path.cwd()
table_dir = cwd.parent / 'train val test tables'
model_dir = cwd.parent / 'models'

In [20]:
# LOAD TRAIN VAL TABLES

# loop through and read tables
table_dfs = []
for table in table_dir.rglob('*'):
    if table.name == 'test.csv':
        continue
    table_dfs.append(pd.read_csv(table, index_col=0))

# concat segmented dfs to train val sets
train_df = pd.concat(table_dfs[0:-2])
val_df = pd.concat(table_dfs[-2:])

len(pd.concat(table_dfs)), len(train_df), len(val_df)

(438, 342, 96)

In [ ]:
# FILTER FOR CHOSEN FEATURES

# combine dfs while tracking what values came from each for masking
df = pd.concat([train_df, val_df], keys=['train', 'val'])

df = df[IN_FEATURES + OUT_FEATURES]
df.dropna(inplace=True)

In [22]:
#  MASK BAD DATA
fluxes = IN_FEATURES[2:]

# mask unbounded results
df = df[~df[IN_FEATURES].eq(-99).any(axis=1)]

# replace negative fluxes with 0
df[fluxes] = df[fluxes].clip(lower=0)

# split dfs again
train_df = df.xs('train')
val_df = df.xs('val')

len(df), len(train_df), len(val_df)

(433, 337, 96)

In [ ]:
# SCALE DATA

scaler = RobustScaler().set_output(transform='pandas')
fluxes = IN_FEATURES[2:]

train_df[fluxes] = np.log1p(train_df[fluxes])
val_df[fluxes] = np.log1p(val_df[fluxes])

X_train_df = scaler.fit_transform(train_df[IN_FEATURES])
X_val_df = scaler.transform(val_df[IN_FEATURES])

stats_df1 = pd.DataFrame(
    {
        'Median_Train': X_train_df.median(),
        'IQR_Train': X_train_df.quantile(0.75) - X_train_df.quantile(0.25),
    }
)
stats_df2 = pd.DataFrame(
    {
        'Median_Val': X_val_df.median(),
        'IQR_Val': X_val_df.quantile(0.75) - X_val_df.quantile(0.25),
    }
)

stats = pd.concat([stats_df1, stats_df2], axis=1).round(4)
print(stats)

print('\nMax values in Train (scaled):')
print(X_train_df.max())

print('\nMax values in Validation (scaled):')
print(X_val_df.max())

      Median_Train  IQR_Train  Median_Val  IQR_Val
beta           0.0        1.0      0.0304   0.9775
Av             0.0        1.0     -0.1083   1.3120
F115           0.0        1.0     -0.0684   0.7868
F150           0.0        1.0     -0.1229   0.7494
F277           0.0        1.0     -0.3033   1.0540
F444           0.0        1.0     -0.3805   1.0642

Max values in Train (scaled):
beta    2.275399
Av      2.366021
F115    2.309122
F150    2.435038
F277    3.661168
F444    3.198234
dtype: float64

Max values in Validation (scaled):
beta    1.836095
Av      2.335162
F115    1.273037
F150    1.278735
F277    2.085680
F444    1.974460
dtype: float64


In [24]:
# CONVERT TO TENSORS AND DATASETS

X_train = torch.tensor(train_df[IN_FEATURES].values)
X_val = torch.tensor(val_df[IN_FEATURES].values)
y_train = torch.tensor(train_df['flag'].values)
y_val = torch.tensor(val_df['flag'].values)

train_data = torch.utils.data.TensorDataset(X_train, y_train)
val_data = torch.utils.data.TensorDataset(X_val, y_val)

In [ ]:
# FUNCTION TO DEFINE MODEL


def define_model(trial):
    # We optimize the number of layers, hidden untis and dropout ratio in each layer.
    n_layers = trial.suggest_int('n_layers', 1, 3)
    layers = []

    in_features = len(IN_FEATURES)
    for i in range(n_layers):
        out_features = trial.suggest_int(f'n_units_l{i}', 4, 128)
        layers.append(nn.Linear(in_features, out_features, bias=False))
        layers.append(nn.BatchNorm1d(out_features))
        layers.append(nn.ReLU())
        p = trial.suggest_float(f'dropout_l{i}', 0.2, 0.5)
        layers.append(nn.Dropout(p))

        in_features = out_features
    layers.append(nn.Linear(in_features, 2 * len(OUT_FEATURES)))

    return nn.Sequential(*layers)

In [ ]:
# TRAINING LOOP FUNCTION


def objective(trial):
    model = define_model(trial).to(DEVICE)

    # optimizer and learning rate
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    optimizer_name = trial.suggest_categorical(
        'optimizer', ['Adam', 'SGD', 'AdamW']
    )

    kwargs = {'lr': lr}

    if optimizer_name == 'AdamW':
        kwargs['weight_decay'] = trial.suggest_float(
            'adamw_weight_decay', 1e-5, 1e-1, log=True
        )
    elif optimizer_name == 'SGD':
        kwargs['momentum'] = trial.suggest_float('sgd_momentum', 0.0, 0.99)

    optimizer = getattr(torch.optim, optimizer_name)(
        model.parameters(), **kwargs
    )

    # weights and loss function
    w1 = trial.suggest_float('w1', 1, 3)
    w2 = trial.suggest_float('w2', 1, 3)
    weights = torch.tensor([w1, w2]).to(DEVICE)
    loss_func = nn.CrossEntropyLoss(weight=weights)

    # batch size and loaders
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    train_loader = torch.utils.data.DataLoader(
        train_data, batch_size=batch_size, shuffle=True, drop_last=True
    )
    val_loader = torch.utils.data.DataLoader(
        val_data, batch_size=batch_size, shuffle=False
    )

    # early stopper
    stopper = ut.EarlyStopper(patience=PATIENCE, min_delta=0.0005)

    for epoch in range(EPOCHS):
        # Training Loop ---------------------------------------------------------------------------------------
        model.train()
        for data, labels in train_loader:
            data, labels = data.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            if data.ndim == 1:
                data = data.unsqueeze(1)
            data = data.float()
            outputs = model(data)
            labels = labels.long()  # for calssification
            loss = loss_func(outputs, labels)

            loss.backward()
            optimizer.step()

        # Validation Loop --------------------------------------------------------------------------------------
        model.eval()
        running_loss = 0
        correct = 0
        with torch.no_grad():
            for data, labels in val_loader:
                data, labels = data.to(DEVICE), labels.to(DEVICE)

                if data.ndim == 1:
                    data = data.unsqueeze(1)
                data = data.float()

                outputs = model(data)
                labels = labels.long()  # for calssification

                loss = loss_func(outputs, labels)
                running_loss += loss.item()

                predictions = torch.argmax(outputs, dim=1)
                correct += (predictions == labels).float().sum()

            # compute accuracy to report to optuna and check for pruning
            accuracy = correct / len(val_data)
            trial.report(accuracy, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            # track loss once per epoch to check early stopper
            val_loss = running_loss / len(val_loader)
            stopper(val_loss)
            if stopper.early_stop:
                break

    return accuracy

In [ ]:
# TRAINS MODEL

if OPTIMIZE:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    sampler = optuna.samplers.GPSampler(seed=SEED)

    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
    )
    study.optimize(objective, n_trials=NUM_TRIALS, show_progress_bar=True)

    print('Best trial:')
    trial = study.best_trial
    print(f'  Accuracy: {trial.value}')
    print('  Params: ')
    for key, value in trial.params.items():
        print(f'    {key}: {value}')

  0%|          | 0/20 [00:00<?, ?it/s]

Best trial:
  Accuracy: 0.7291666865348816
  Params: 
    n_layers: 2
    n_units_l0: 4
    dropout_l0: 0.2482424154252496
    n_units_l1: 72
    dropout_l1: 0.407568559307808
    lr: 0.004053638704742309
    optimizer: SGD
    sgd_momentum: 0.32214570117767505
    w1: 2.4929828102360485
    w2: 2.2992657980944293
    batch_size: 16


In [28]:
# LOG OPTIMIZATION

if OPTIMIZE:
    logged_params = trial.params.copy()
    logged_params['accuracy'] = trial.value
    logged_params['comment'] = COMMENT
    logged_params['in_features'] = IN_FEATURES
    logged_params['out_features'] = OUT_FEATURES
    logged_params['seed'] = SEED
    logged_params['epochs'] = EPOCHS
    logged_params['num_trials'] = NUM_TRIALS
    logged_params['patience'] = PATIENCE

    logger = ut.RunLogger(filepath=cwd.parent / 'logs' / 'fc_log.json')
    logger.log_run(logged_params)